## Run snow cover mapping RF model
Author: Justin Pflug and Emma Boudreau

Option to run Composite model maps (model trained in the forest and model trained out of the forest) and Base model (single trained model)

In [1]:
# notebook I've been working on 6/10
# load dependencies
import joblib
import os
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import rioxarray as rxr
import datetime
import geopandas as gpd
from working_sca_funcs import create_binary_chm
import xarray as xr

## SET USER DEFINITIONS

In [ ]:
# location of the Planet data
focus_year = 2022
name = 'BUDD'
model = 'V9'
processed_direc = f'/home/etboud/projects/data/rerun/{name}/{model}/'
data_direc = f'/data0/images/planet/emma/planet/{name}/'

# specify if the output snow cover maps should be saved or not (1 = save, 0 = do not save)
saveData = 1

# specify the model used to classify snow presence, snow absence, and artifacts
# model = joblib.load('/home/etboud/projects/data/training_data/aso_BUDD_20220518_binary_V5.joblib') # lidar trained
# model = joblib.load('/home/etboud/projects/data/training_data/manual_BUDD_2022_3class_V7.joblib') # manual trained
model_can = joblib.load('/home/etboud/projects/data/training_data/aso_BUDD_20220518_binary_canopy_V9.joblib')
model_open = joblib.load('/home/etboud/projects/data/training_data/aso_BUDD_20220518_binary_open_V9.joblib')

# specify the directories and the indices of which PS scenes to filter, if any
subdirecs = sorted([d for d in glob.glob(data_direc + str(focus_year) + '*') if os.path.isdir(d)])
# Filter any scenes you wish to be excluded
filtered_scenes = []
# reference file containing the maximum extents
ref_file = f'/data0/images/planet/emma/planet/{name}/' + '20230423_175100_99_242e/a514a5df-5610-45c4-8498-a774361cb492/PSScene/20230423_175100_99_242e_3B_AnalyticMS_SR_clip.tif' # BUDD


# # Uncomment this if you are running for one mode (not composite)
# ref_file = rxr.open_rasterio(ref_file)
# mask = ref_file.values
# mask = np.sum(mask,axis=0)
# ref_file = ref_file.isel(band=0)
# ref_file.values = mask
# ref_file.plot()


In [3]:
# For open and canopy cover model
# Create basin shapefile
BS = f'/home/etboud/projects/data/basins/{name}/{name}_4326.geojson'
basin = gpd.read_file(BS)
basin = basin.to_crs('EPSG:32611')

# canopy height model
CHM = '/home/etboud/projects/data/CHM/USCATB20140827_chm_3p0m.tif'
chm_mask, mean, max, fcan = create_binary_chm(CHM, basin)


ref_file = rxr.open_rasterio(ref_file).rio.clip(basin.geometry)
open_ref = ref_file.copy()
open_ref.values = np.where(chm_mask[0], open_ref, 0)
canop_ref = ref_file - open_ref
canop_ref.attrs = open_ref.attrs
canop_ref.rio.set_nodata(0)


open_mask = open_ref.values
open_mask = np.sum(open_mask, axis=0)
open_ref = open_ref.isel(band=0)
open_ref.values = open_mask
# Create a canopy reference where 0 replaces open

canop_mask = canop_ref.values
canop_mask = np.sum(canop_mask, axis=0)
canop_ref = canop_ref.isel(band=0)
canop_ref.values = canop_mask


In [ ]:
# # looking at scenes to be filtered?
# for dCount,direcc in enumerate(subdirecs):
#     if dCount in filtered_scenes:
#         fname = glob.glob(direcc+'/*/PSScene/*SR_clip.tif')[0]
        
#         outfile_0 = fname.split('/')[-1].split('_')
#         outfile = processed_direc+outfile_0[0]+'_'+outfile_0[1]+'_SCA.tif'
#         print(outfile)
        

#         # process the image and plot the image versus the snow cover map for visual inspection
#         rgb_image = rxr.open_rasterio(fname)
#         _,_,_,_,rgb_image = calc_rgb(rgb_image)
#         fg,ax = plt.subplots(1,1)
#         ax = np.ravel(ax)
#         ax[0].imshow(rgb_image,cmap='gray')
#         ax[0].set_title(outfile_0[0]+outfile_0[1])
        
        

## FUNCTIONS

In [ ]:
# function for classifying snow cover for single model
def run_sca_prediction_band_selfClassify(f_raster, file_out, model,saveData,ref_file,mask):
        
    ds = rxr.open_rasterio(f_raster)
    ds = ds.rio.reproject_match(ref_file)
    arr = ds.values
    arr_sum = np.sum(arr,axis=0)
    
    print("Image dimension:".format(), arr.shape)  # 
    X_img = pd.DataFrame(arr.reshape([4,-1]).T)
    X_img.columns = ['b','g','r','nir']
    X_img
    y_img = model.predict(X_img)
    
    out_img = pd.DataFrame()
    out_img['label'] = y_img
    
    # Reshape our classification map
    img_prediction = out_img['label'].to_numpy().reshape(arr[0,:, :].shape)
    # put "no-data" classifications in the cells that are just cut off from the obs
    img_prediction[(arr_sum == 0) & (mask > 0)] = 2

    if saveData:
        ds_save = ref_file
        ds_save.attrs['long_name'] = ('classification')
        ds_save.values = img_prediction
        ds_save.rio.to_raster(file_out)
            
    return img_prediction

# calculate the rgb bands and normalize radiances
# see 1_classify_train_model.ipynb
def calc_rgb(ds):
    # Selecting RGB bands
    blue_band = ds.isel(band=0)
    green_band = ds.isel(band=1)
    red_band = ds.isel(band=2)
    nir_band = ds.isel(band=3)
    
    # normalize
    maxval = green_band.max().values
    minval = green_band.min().values
    red_norm = (red_band - minval) / (maxval - minval)
    green_norm = (green_band - minval) / (maxval - minval)
    blue_norm = (blue_band - minval) / (maxval - minval)
    green_norm = green_norm.where(red_norm <= 1,1)
    blue_norm = blue_norm.where(red_norm <= 1,1)
    red_norm = red_norm.where(red_norm <= 1,1)

    red_band = red_band.values
    green_band = green_band.values
    blue_band = blue_band.values
    nir_band = nir_band.values
    
    # Stack normalized bands to create RGB image
    rgb_image = np.stack([red_norm, green_norm, blue_norm], axis=-1)
    return red_band,green_band,blue_band,nir_band,rgb_image

### Test function works for in an out of canopy classification

In [ ]:
# function for classifying snow cover using "model"
def run_composite_sca_prediction(f_raster, file_out, model,saveData,ref_file,mask):
        
    ds = rxr.open_rasterio(f_raster)
    ds = ds.rio.reproject_match(ref_file)
    
    zeros = np.zeros((1, ds.shape[1], ds.shape[2]))
    mask_bool = mask > 0
    mask_num = ds.where(mask_bool)
    clipped_ds = mask_num + zeros
    arr = clipped_ds.values
    # arr = ds.values
    arr_sum = np.sum(arr,axis=0)
    
    print("Image dimension:".format(), arr.shape)  # 
    X_img = pd.DataFrame(arr.reshape([4,-1]).T)
    X_img.columns = ['b','g','r','nir']
    X_img
    y_img = model.predict(X_img)
    
    out_img = pd.DataFrame()
    out_img['label'] = y_img
    
    # Reshape our classification map
    img_prediction = out_img['label'].to_numpy().reshape(arr[0,:, :].shape)
    # put "no-data" classifications in the cells that are just cut off from the obs
    img_prediction[(arr_sum == 0) & (mask > 0)] = 2

    # if saveData:
    #     ds_save = ref_file
    #     ds_save.attrs['long_name'] = ('classification')
    #     ds_save.values = img_prediction
    #     ds_save.rio.to_raster(file_out)
            
    return img_prediction

## SCRIPTS FOR PROCESSING SNOW COVER USING THE TRAINED RF MODEL

In [ ]:
ref_file = f'/data0/images/planet/emma/planet/{name}/' + '20230423_175100_99_242e/a514a5df-5610-45c4-8498-a774361cb492/PSScene/20230423_175100_99_242e_3B_AnalyticMS_SR_clip.tif' # BUDD
reference_file = rxr.open_rasterio(ref_file) #.rio.clip(basin.geometry)
mask = reference_file.values
mask = np.sum(mask,axis=0)
reference_file = reference_file.isel(band=0)
reference_file.values = mask

#### processing scenes for composite model

In [ ]:
# For composite model
for dCount,direcc in enumerate(subdirecs):
    if dCount not in filtered_scenes:
        fname = glob.glob(direcc+'/*/PSScene/*SR_clip.tif')[0]
        
        outfile_0 = fname.split('/')[-1].split('_')
        outfile = processed_direc+outfile_0[0]+'_'+outfile_0[1]+'_SCA.tif'
        print(outfile)
            
        classified_can = run_composite_sca_prediction(fname,outfile,model_can,saveData,canop_ref,canop_mask)
        classified_open = run_composite_sca_prediction(fname,outfile,model_open,saveData,open_ref,open_mask)
        classified = classified_can + classified_open
        classified = np.where(classified == 1, 0, classified)
        classified = np.where(classified == 2, 1, classified)
        
        # Save the classification result
        ds_save = reference_file
        ds_save.attrs['long_name'] = ('classification')
        ds_save.values = classified
        ds_save.rio.to_raster(outfile)
        
        # process the image and plot the image versus the snow cover map for visual inspection
        rgb_image = rxr.open_rasterio(fname)
        _,_,_,_,rgb_image = calc_rgb(rgb_image)
        fg,ax = plt.subplots(1,4)
        ax = np.ravel(ax)
        ax[0].imshow(rgb_image,cmap='gray')
        ax[1].imshow(classified_can,vmin=0,vmax=2,interpolation='none')
        ax[2].imshow(classified_open,vmin=0,vmax=2, interpolation='none')
        ax[3].imshow(classified,vmin=0,vmax=2, interpolation='none')
        ax[0].set_title(outfile_0[0])
        ax[0].set_xticks([])
        ax[1].set_xticks([])
        ax[2].set_xticks([])
        ax[3].set_xticks([])
        ax[0].set_yticks([])
        ax[1].set_yticks([])
        ax[2].set_yticks([])
        ax[3].set_yticks([])

#### processing scenes for base model

In [ ]:
for dCount,direcc in enumerate(subdirecs):
    if dCount not in filtered_scenes:
        fname = glob.glob(direcc+'/*/PSScene/*SR_clip.tif')[0]
        
        outfile_0 = fname.split('/')[-1].split('_')
        outfile = processed_direc+outfile_0[0]+'_'+outfile_0[1]+'_SCA.tif'
        print(outfile)
            
        classified = run_sca_prediction_band_selfClassify(fname,outfile,model,saveData,ref_file,mask)

        # process the image and plot the image versus the snow cover map for visual inspection
        rgb_image = rxr.open_rasterio(fname)
        _,_,_,_,rgb_image = calc_rgb(rgb_image)
        fg,ax = plt.subplots(1,2)
        ax = np.ravel(ax)
        ax[0].imshow(rgb_image,cmap='gray')
        ax[1].imshow(classified,vmin=0,vmax=2,interpolation='none')
        ax[0].set_title(outfile_0[0])
        ax[0].set_xticks([])
        ax[1].set_xticks([])
        ax[0].set_yticks([])
        ax[1].set_yticks([])

    # if dCount == 2:
    #     break

In [ ]:
# checking the output
# dirr = '/home/etboud/projects/data/planet/processed_SCA/'

# for d in glob.glob(dirr + str(focus_year) + '*'):
#     cl= rxr.open_rasterio(d)
#     #print(d)
#     fig, ax = plt.subplots()
#     ax.imshow(cl[0],vmin=0,vmax=2,interpolation='none')
#     ax.set_title(d.split('/')[-1])